# solvecdp

> Lightweight Chrome Debug Protocol (CDP) client for solveit

fastcdp provides an async Python client for the [Chrome DevTools Protocol](https://chromedevtools.github.io/devtools-protocol/) (CDP) over solveit's js bridge. It exposes every CDP domain as a Python attribute with auto-generated signatures and docstrings — e.g. `await cdp.page.navigate(url=...)`.

It includes event subscription via `cdp.on()`/`cdp.wait_event()`, navigation helpers (`goto`, `wait_for_selector`, `wait_for`), screenshot capture, and accessibility tree access. A `cdp_search` utility lets you search CDP commands by name or description. For use inside [safepyrun](https://github.com/AnswerDotAI/safepyrun) sandboxes, `cdp_yolo()` registers all CDP classes.

## Installation

You must install the [solveit-chrome extension](https://github.com/AnswerDotAI/solveit-chrome) before using solvecdp.

Solveit already has solvecdp installed, but if you want to install the latest you can get it from [pypi][pypi]:


```sh
$ pip install solvecdp
```

[pypi]: https://pypi.org/project/solvecdp/

## How to use

In [ ]:
from solvecdp import *

### The JsCDP class

Every CDP domain is available as an attribute with auto-generated signatures. You can search for commands with `cdp_search`:

In [ ]:
cdp_search('screenshot')

"Emulation.setVisibleSize: Resizes the frame/viewport of the page. Note that this does not affect the frame's container\n(e.g. browser window). Can \nHeadlessExperimental.beginFrame: Sends a BeginFrame to the target and returns when the frame was completed. Optionally captures a\nscreenshot from the res\n  evt Overlay.screenshotRequested: Fired when user asks to capture screenshot of some area on the page.\nPage.captureScreenshot: Capture page screenshot."

Create a new page:

In [ ]:
jc = await JsCDP.new()

Go to a page:

In [ ]:
await jc.goto('https://httpbin.org/forms/post')

Eval js:

In [ ]:
await jc.eval('document.title')

'6. httpbin.org/forms/post'

Or you can `wait_for` any js expression to be truthy, and have it returned:

In [ ]:
await jc.wait_for('document.title')

'6. httpbin.org/forms/post'

Take a screenshot of the page:

In [ ]:
img = await jc.screenshot()

Clean up when done:

In [ ]:
await jc.close()

See `JsCDP` docs for full details.

## Filling forms

In [ ]:
page = await JsCDP.new(url='https://httpbin.org/forms/post')

For finding elements to interact with, use `ax_tree`:

In [ ]:
root = await page.ax_tree()
print(str(root)[:300])

- **RootWebArea** "6. httpbin.org/forms/post" `focusable=True` `url=https://httpbin.org/forms/post` [#14]
  - **LabelText** "" [#20]
    - **StaticText** "Customer name: " [#62]
      - **InlineTextBox** "Customer name: "
    - **textbox** "Customer name: " `focusable=True` `editable=plaintext` `set


`find` and `find_id` are used to identify elements in the tree:

In [ ]:
nmid = root.find_id('textbox', 'Customer name')
nmid

2

You can use regular CDP methods, or one of the provided shortcuts:

In [ ]:
await page.fill_text(nmid, 'Jeremy Howard')
await page.click(root.find_id('radio', 'Large'))
await page.js_node_run('this.value = "18:30"', root.find_id('InputTime', 'delivery time'));

You can use `click` to click a button, or `click_and_wait` to wait for the next page to load:

In [ ]:
await page.click_and_wait(root.find_id('button', 'Submit order'))

In [ ]:
await page.close()

To allow LLMs like solveit with safepyrun to access solvecdp, use:

```python
cdp_yolo()
```

Then use a prompt such as:

> Try using pyrun to create a `page_` JsCDP object, then goto `<url>`, fill it out, read it to check it's filled correctly, then submit it, and see what you get back. Don't use find_id - you can get all the ids at once with ax_tree (don't truncate the result of it). Don't add extra waits etc - solvecdp handles it automatically. IDs can change so be sure to use the ax_tree IDs you read.